# Smart Waste: Klasifikasi Gambar Sampah (Organic / Paper / Plastic)

Notebook ini melakukan **semua** bagian Machine Learning untuk tes Bilcode:

| Langkah | Apa yang terjadi | Hasil |
|---|---|---|
| 1-3 | Ambil dataset Kaggle, cek isinya | daftar folder + jumlah gambar |
| 4 | Susun jadi 3 kelas, buang duplikat, bagi train/val/test | folder `dataset/` |
| 5-8 | Training model (transfer learning MobileNetV2) | model terlatih |
| 9 | Evaluasi di data test | accuracy, confusion matrix, classification report |
| 10 | Simpan model | `waste_model.keras` + `model_meta.json` |
| 11-12 | Coba foto sendiri, unduh hasil | file untuk folder `ml/` di repo |

**Sebelum mulai:** menu *Runtime → Change runtime type → T4 GPU*, lalu jalankan sel dari atas ke bawah (Shift+Enter).

### Ide dasarnya (untuk interview)
- **Model** = fungsi yang menerima gambar dan mengeluarkan 3 angka (peluang tiap kelas).
- **Training** = menyesuaikan angka-angka di dalam model supaya tebakannya cocok dengan label pada data latih.
- **Transfer learning** = kita tidak melatih dari nol. MobileNetV2 sudah dilatih Google pada jutaan gambar, jadi sudah "paham" tepi, tekstur, bentuk. Kita hanya menambah lapisan kecil di ujung untuk 3 kelas kita.
- **Train / Val / Test**: *train* untuk belajar, *val* untuk memantau selama training, *test* disimpan sampai akhir untuk nilai jujur.

## Langkah 0: Import dan cek lingkungan

In [ ]:
import os, io, json, shutil, random, hashlib, datetime
from collections import Counter
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import tensorflow as tf
from tensorflow import keras
from sklearn.metrics import classification_report, confusion_matrix

print("TensorFlow:", tf.__version__, "| Keras:", keras.__version__)
gpus = tf.config.list_physical_devices("GPU")
print("GPU:", gpus if gpus else "TIDAK ADA -> Runtime > Change runtime type > T4 GPU")

## Langkah 1: Konfigurasi

Semua pengaturan ada di sini. Biasanya cukup dibiarkan.

**Soal dataset (penting):** dataset `asdasdasasdas/garbage-classification` (TrashNet) hanya punya 6 kelas: cardboard, glass, metal, paper, plastic, trash. **Tidak ada Organic.** Karena itu kita pakai dua sumber:

- **A**: dataset milikmu (TrashNet), dipakai untuk `paper` dan `plastic`.
- **B**: `mostafaabla/garbage-classification` (12 kelas), folder `biological` dipakai sebagai `organic`. Folder `paper` dan `plastic` miliknya ikut dipakai supaya foto organik dan non-organik berasal dari gaya foto yang beragam.

Folder lain (glass, metal, clothes, dst.) diabaikan.

In [ ]:
SEED = 42
IMG_SIZE = (224, 224)          # ukuran input MobileNetV2
BATCH_SIZE = 32
MAX_PER_CLASS = 800            # batas gambar per kelas (supaya seimbang dan training cepat)
SPLIT = (0.70, 0.15, 0.15)     # train / val / test
EPOCHS_HEAD = 10               # tahap 1: hanya melatih lapisan baru
EPOCHS_FINE = 5                # tahap 2: fine-tuning ujung MobileNetV2 (0 = lewati)
FINE_TUNE_LAST_LAYERS = 30
WEIGHTS = "imagenet"           # bobot awal MobileNetV2
INCLUDE_CARDBOARD_AS_PAPER = True   # kardus dihitung sebagai Paper Waste

# nama folder di dataset sumber (huruf kecil) -> kelas target
CLASS_MAP = {
    "organic": {"biological", "organic", "organic waste", "organic_waste"},
    "paper":   {"paper"},
    "plastic": {"plastic"},
}
if INCLUDE_CARDBOARD_AS_PAPER:
    CLASS_MAP["paper"].add("cardboard")

# nama yang dikirim API ke backend (harus sama dengan LABEL_TO_CATEGORY di NestJS)
DISPLAY_NAMES = {"organic": "Organic Waste", "paper": "Paper Waste", "plastic": "Plastic Waste"}

KAGGLE_SOURCES = {
    "A_trashnet": "asdasdasasdas/garbage-classification",
    "B_12class":  "mostafaabla/garbage-classification",
}

WORK = Path("/content") if Path("/content").exists() else Path.cwd() / "work"
DATA_DIR = WORK / "dataset"            # hasil susunan train/val/test
OUT_DIR = WORK / "output"              # semua file yang akan diunduh
EVAL_DIR = OUT_DIR / "evaluation"
for d in (OUT_DIR, EVAL_DIR):
    d.mkdir(parents=True, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
keras.utils.set_random_seed(SEED)
SOURCES = {}   # diisi di langkah 2

## Langkah 2: Unduh dataset dari Kaggle

`kagglehub` mengunduh dataset publik langsung ke Colab (sekitar beberapa ratus MB, tunggu sebentar).

Kalau gagal (misalnya minta login), jalankan `kagglehub.login()` lalu ulangi sel ini. Cara manual: unduh zip dari halaman Kaggle, unggah ke panel file Colab, `!unzip`, lalu isi `SOURCES["A_trashnet"] = Path("/content/folder-hasil-unzip")`.

In [ ]:
!pip install -q kagglehub
import kagglehub

SOURCES = {}
for name, slug in KAGGLE_SOURCES.items():
    try:
        SOURCES[name] = Path(kagglehub.dataset_download(slug))
        print("OK    ", name, "->", SOURCES[name])
    except Exception as e:
        print("GAGAL ", name, slug, "->", e)

## Langkah 3: Cek isi dataset

Sel ini menelusuri semua folder berisi gambar, lalu mencocokkan **nama folder** dengan `CLASS_MAP`. Periksa tabelnya: jumlah per kelas harus masuk akal, dan folder yang tidak dipakai muncul di daftar "diabaikan".

In [ ]:
IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

def scan_sources(sources):
    found = {t: [] for t in CLASS_MAP}          # kelas -> [(path, nama_sumber)]
    ignored = Counter()
    for src_name, root in sources.items():
        for folder, _, files in os.walk(root):
            imgs = [f for f in files if Path(f).suffix.lower() in IMG_EXT]
            if not imgs:
                continue
            fname = Path(folder).name.lower()
            target = next((t for t, names in CLASS_MAP.items() if fname in names), None)
            if target:
                found[target] += [(Path(folder) / f, src_name) for f in imgs]
            else:
                ignored[f"{src_name}/{fname}"] += len(imgs)
    return found, ignored

assert SOURCES, "SOURCES kosong: jalankan Langkah 2 (atau isi manual)."
found, ignored = scan_sources(SOURCES)

print(f"{'kelas':<10}{'total':>7}   per sumber")
for cls, items in found.items():
    per_src = Counter(s for _, s in items)
    print(f"{cls:<10}{len(items):>7}   {dict(per_src)}")
print("\nDiabaikan (tidak dipakai):", dict(ignored))

missing = [c for c, items in found.items() if not items]
if missing:
    raise RuntimeError(
        f"Kelas {missing} tidak ditemukan. Lihat daftar 'Diabaikan' di atas: "
        "kalau nama foldernya berbeda (misal 'organik'), tambahkan ke CLASS_MAP di Langkah 1."
    )

## Langkah 4: Susun dataset 3 kelas dan bagi train / val / test

Yang dilakukan sel ini:
1. Acak urutan gambar (seed tetap, hasil bisa diulang).
2. **Buang duplikat** (hash MD5). Dataset Kaggle sering berisi gambar kembar; kalau satu kembar masuk train dan satunya masuk test, nilai test jadi palsu tinggi (*data leakage*).
3. Buang file gambar rusak.
4. Ambil maksimal `MAX_PER_CLASS` per kelas, lalu bagi 70/15/15 **per kelas**.
5. Salin ke folder `dataset/{train,val,test}/{organic,paper,plastic}`.

In [ ]:
def md5_of(path):
    h = hashlib.md5()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

def is_valid_image(path):
    try:
        with Image.open(path) as im:
            im.verify()
        return True
    except Exception:
        return False

rng = random.Random(SEED)
if DATA_DIR.exists():
    shutil.rmtree(DATA_DIR)

seen, summary = set(), {}
for cls, items in found.items():
    items = list(items)
    rng.shuffle(items)
    chosen, dup, bad = [], 0, 0
    for path, src in items:
        if len(chosen) >= MAX_PER_CLASS:
            break
        h = md5_of(path)
        if h in seen:
            dup += 1
            continue
        if not is_valid_image(path):
            bad += 1
            continue
        seen.add(h)
        chosen.append((path, src))

    n = len(chosen)
    n_tr, n_va = int(n * SPLIT[0]), int(n * SPLIT[1])
    parts = {"train": chosen[:n_tr], "val": chosen[n_tr:n_tr + n_va], "test": chosen[n_tr + n_va:]}
    for split, rows in parts.items():
        out = DATA_DIR / split / cls
        out.mkdir(parents=True, exist_ok=True)
        for i, (p, src) in enumerate(rows):
            shutil.copy2(p, out / f"{cls}_{src}_{i:04d}{p.suffix.lower()}")

    summary[cls] = {
        "train": len(parts["train"]), "val": len(parts["val"]), "test": len(parts["test"]),
        "total": n, "duplicates_removed": dup, "corrupt_removed": bad,
        "sources": dict(Counter(s for _, s in chosen)),
    }

total = sum(v["total"] for v in summary.values())
print(f"Total gambar dipakai: {total} (syarat tes: minimal 150)\n")
print("| Kelas | Train | Val | Test | Total | Sumber | Duplikat dibuang |")
print("|---|---|---|---|---|---|---|")
for cls, v in summary.items():
    print(f"| {cls} | {v['train']} | {v['val']} | {v['test']} | {v['total']} | {v['sources']} | {v['duplicates_removed']} |")

assert total >= 150, "Dataset kurang dari 150 gambar."
(OUT_DIR / "dataset_summary.json").write_text(json.dumps(summary, indent=2))

## Langkah 5: Muat data ke TensorFlow

`image_dataset_from_directory` membaca folder dan memakai **nama folder sebagai label**, diurutkan abjad: `organic=0, paper=1, plastic=2`. Urutan ini dicatat otomatis ke `model_meta.json` supaya FastAPI tidak salah menamai kelas.

Data val dan test **tidak diacak** (`shuffle=False`) supaya urutan prediksi cocok dengan urutan label saat evaluasi.

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

def load(split, shuffle):
    return keras.utils.image_dataset_from_directory(
        DATA_DIR / split, image_size=IMG_SIZE, batch_size=BATCH_SIZE,
        shuffle=shuffle, seed=SEED,
    )

train_ds, val_ds, test_ds = load("train", True), load("val", False), load("test", False)
class_names = train_ds.class_names
print("Urutan kelas:", class_names)

train_ds = train_ds.prefetch(AUTOTUNE)
val_ds = val_ds.cache().prefetch(AUTOTUNE)
test_ds = test_ds.prefetch(AUTOTUNE)

# lihat sampel gambar training
plt.figure(figsize=(9, 5))
for images, labels in train_ds.take(1):
    for i in range(min(8, len(images))):
        plt.subplot(2, 4, i + 1)
        plt.imshow(images[i].numpy().astype("uint8"))
        plt.title(class_names[int(labels[i])])
        plt.axis("off")
plt.tight_layout()
plt.show()

## Langkah 6: Bangun model

Susunan lapisan (dari input ke output):
1. **Augmentasi** (flip, rotasi, zoom, kontras): variasi acak agar model tidak menghafal. Aktif **hanya saat training**.
2. **Rescaling**: ubah piksel 0..255 menjadi -1..1, sesuai yang diharapkan MobileNetV2. Ditaruh **di dalam model**, jadi FastAPI cukup mengirim piksel mentah 0..255 dan tidak ada risiko preprocessing beda.
3. **MobileNetV2** (dibekukan di tahap 1): pengekstrak fitur.
4. **GlobalAveragePooling2D**: meringkas fitur menjadi satu vektor.
5. **Dropout 0.3**: mengacak sebagian neuron agar tidak overfit.
6. **Dense(3, softmax)**: 3 peluang yang jumlahnya 1.

In [ ]:
def build_model(num_classes):
    augmentation = keras.Sequential([
        keras.layers.RandomFlip("horizontal"),
        keras.layers.RandomRotation(0.1),
        keras.layers.RandomZoom(0.15),
        keras.layers.RandomContrast(0.1),
    ], name="augmentation")

    base = keras.applications.MobileNetV2(
        input_shape=IMG_SIZE + (3,), include_top=False, weights=WEIGHTS,
    )
    base.trainable = False

    inputs = keras.Input(shape=IMG_SIZE + (3,))
    x = augmentation(inputs)
    x = keras.layers.Rescaling(1.0 / 127.5, offset=-1)(x)
    x = base(x, training=False)
    x = keras.layers.GlobalAveragePooling2D()(x)
    x = keras.layers.Dropout(0.3)(x)
    outputs = keras.layers.Dense(num_classes, activation="softmax")(x)
    return keras.Model(inputs, outputs, name="waste_classifier"), base

model, base = build_model(len(class_names))
model.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)
model.summary()

## Langkah 7: Training tahap 1 (melatih lapisan baru)

- **Epoch** = satu putaran penuh melihat seluruh data latih.
- **EarlyStopping**: berhenti kalau `val_accuracy` tidak membaik 3 epoch, lalu kembali ke bobot terbaik.
- Yang perlu diperhatikan: `accuracy` (train) dan `val_accuracy` sama-sama naik. Kalau train jauh di atas val, itu tanda *overfitting*.

In [ ]:
def make_callbacks():
    return [
        keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=3, restore_best_weights=True),
        keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, min_lr=1e-6),
    ]

hist_head = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_HEAD, callbacks=make_callbacks(), shuffle=False)

## Langkah 8: Fine-tuning (tahap 2)

Sekarang `FINE_TUNE_LAST_LAYERS` lapisan terakhir MobileNetV2 dibuka dan dilatih dengan laju belajar sangat kecil (`1e-5`), supaya fitur umumnya disesuaikan dengan gambar sampah tanpa merusak yang sudah dipelajari. Biasanya menambah beberapa persen akurasi.

In [ ]:
hist_fine = None
if EPOCHS_FINE > 0:
    base.trainable = True
    for layer in base.layers[:-FINE_TUNE_LAST_LAYERS]:
        layer.trainable = False
    model.compile(
        optimizer=keras.optimizers.Adam(1e-5),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    hist_fine = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_FINE, callbacks=make_callbacks(), shuffle=False)
else:
    print("Fine-tuning dilewati.")

In [ ]:
# grafik akurasi dan loss (disimpan untuk laporan)
def merged(key):
    values = list(hist_head.history[key])
    if hist_fine is not None:
        values += list(hist_fine.history[key])
    return values

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, key, title in zip(axes, ["accuracy", "loss"], ["Accuracy", "Loss"]):
    ax.plot(merged(key), label="train")
    ax.plot(merged("val_" + key), label="val")
    if hist_fine is not None:
        ax.axvline(len(hist_head.history[key]) - 0.5, color="gray", linestyle="--", label="mulai fine-tuning")
    ax.set_title(title); ax.set_xlabel("epoch"); ax.legend()
fig.tight_layout()
fig.savefig(EVAL_DIR / "training_curves.png", dpi=150)
plt.show()

## Langkah 9: Evaluasi di data TEST

Data test belum pernah dipakai untuk belajar maupun memilih bobot, jadi inilah angka jujur untuk laporan.

- **Accuracy**: persentase tebakan yang benar. Target tes: **≥ 80%**.
- **Confusion matrix**: baris = kelas sebenarnya, kolom = tebakan model. Angka di diagonal = benar; di luar diagonal = salah (misal plastik ditebak kertas).
- **Precision**: dari semua yang ditebak kelas X, berapa yang benar-benar X.
- **Recall**: dari semua yang sebenarnya X, berapa yang berhasil ditemukan.
- **F1**: rata-rata harmonis precision dan recall.

In [ ]:
y_true = np.concatenate([y.numpy() for _, y in test_ds])
probs = model.predict(test_ds, verbose=0)
y_pred = probs.argmax(axis=1)

test_acc = float((y_true == y_pred).mean())
labels = list(range(len(class_names)))
report = classification_report(y_true, y_pred, labels=labels, target_names=class_names, digits=4, zero_division=0)
cm = confusion_matrix(y_true, y_pred, labels=labels)

print(f"Jumlah gambar test: {len(y_true)}")
print(f"TEST ACCURACY: {test_acc:.2%}")
print("OK: target >= 80% tercapai" if test_acc >= 0.80 else "BELUM 80%: lihat 'Troubleshooting' di akhir notebook")
print("\nClassification report:\n", report)
print("Confusion matrix (baris=sebenarnya, kolom=prediksi):\n", cm)

# gambar confusion matrix
n = len(class_names)
fig, ax = plt.subplots(figsize=(5.5, 5))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(n)); ax.set_xticklabels(class_names, rotation=30, ha="right")
ax.set_yticks(range(n)); ax.set_yticklabels(class_names)
for i in range(n):
    for j in range(n):
        ax.text(j, i, int(cm[i, j]), ha="center", va="center",
                color="white" if cm[i, j] > cm.max() / 2 else "black")
ax.set_xlabel("Prediksi"); ax.set_ylabel("Sebenarnya")
ax.set_title(f"Confusion Matrix (accuracy {test_acc:.2%})")
fig.colorbar(im, ax=ax, fraction=0.046)
fig.tight_layout()
fig.savefig(EVAL_DIR / "confusion_matrix.png", dpi=150)
plt.show()

(EVAL_DIR / "classification_report.txt").write_text(
    f"Test accuracy: {test_acc:.4f}\nTest images: {len(y_true)}\n\n{report}\nConfusion matrix:\n{cm}\n"
)
(EVAL_DIR / "metrics.json").write_text(json.dumps({
    "test_accuracy": test_acc, "test_images": int(len(y_true)),
    "class_names": class_names, "confusion_matrix": cm.tolist(),
}, indent=2))

## Langkah 10: Simpan model dan metadata

Dua file ini yang dipakai FastAPI:
- `waste_model.keras`: arsitektur + bobot.
- `model_meta.json`: urutan kelas, nama tampilan, ukuran gambar, versi TensorFlow. FastAPI membaca urutan kelas dari sini, jadi tidak ada risiko label tertukar.

Sel ini juga memuat ulang model dari file dan membandingkan prediksinya dengan model di memori, untuk memastikan file yang tersimpan benar.

In [ ]:
model_path = OUT_DIR / "waste_model.keras"
model.save(model_path)

meta = {
    "class_names": class_names,
    "display_names": {c: DISPLAY_NAMES[c] for c in class_names},
    "img_size": list(IMG_SIZE),
    "input": "RGB float32 0..255 (Rescaling ada di dalam model)",
    "tensorflow_version": tf.__version__,
    "keras_version": keras.__version__,
    "test_accuracy": test_acc,
    "trained_at": datetime.datetime.now().isoformat(timespec="seconds"),
}
(OUT_DIR / "model_meta.json").write_text(json.dumps(meta, indent=2))

# uji muat ulang
reloaded = keras.models.load_model(model_path)
batch = next(iter(test_ds))[0]
same = np.allclose(model.predict(batch, verbose=0), reloaded.predict(batch, verbose=0), atol=1e-4)
print("Model tersimpan:", model_path, f"({model_path.stat().st_size / 1e6:.1f} MB)")
print("Prediksi model tersimpan sama dengan model asli:", same)
print("\nTempel baris ini ke ml/api/requirements.txt (versi harus sama dengan Colab):")
print(f"tensorflow-cpu=={tf.__version__}")

## Langkah 11: Coba dengan foto sendiri (sangat disarankan)

Foto dataset biasanya bersih, sedangkan foto pengguna sering miring, gelap, dan ramai. Unggah 5 sampai 10 foto sampah asli dari HP-mu (organik, kertas, plastik) dan lihat hasilnya. Preprocessing di sini **sama persis** dengan yang dipakai FastAPI.

In [ ]:
from google.colab import files
from PIL import ImageOps

uploaded = files.upload()
for name, data in uploaded.items():
    img = ImageOps.exif_transpose(Image.open(io.BytesIO(data))).convert("RGB")
    x = np.asarray(img.resize(IMG_SIZE, Image.BILINEAR), dtype="float32")[None, ...]
    p = model.predict(x, verbose=0)[0]
    top = int(p.argmax())
    plt.figure(figsize=(3, 3)); plt.imshow(img); plt.axis("off")
    plt.title(f"{DISPLAY_NAMES[class_names[top]]}\n{p[top]:.0%}")
    plt.show()
    print(name, {DISPLAY_NAMES[c]: round(float(v), 3) for c, v in zip(class_names, p)})

## Langkah 12: Unduh hasil

Sel ini mengemas folder `output/` (model, metadata, evaluasi, ringkasan dataset) menjadi satu zip.

Setelah diunduh dan diekstrak:
- `waste_model.keras` dan `model_meta.json` → `ml/api/model/`
- folder `evaluation/` → `ml/evaluation/`
- `dataset_summary.json` → `ml/dataset/` (isi tabelnya ke `ml/dataset/README.md`)
- notebook ini (File → Download .ipynb) → `ml/notebooks/train_waste_classifier.ipynb`

In [ ]:
from google.colab import files
zip_path = shutil.make_archive(str(WORK / "ml_output"), "zip", OUT_DIR)
print("Zip:", zip_path)
files.download(zip_path)

## Troubleshooting

| Masalah | Penyebab umum | Solusi |
|---|---|---|
| Accuracy test < 80% | Epoch kurang, atau gambar organik terlalu beda dengan lainnya | Naikkan `EPOCHS_HEAD` ke 15 dan `FINE_TUNE_LAST_LAYERS` ke 50; pastikan GPU aktif |
| `train acc` jauh di atas `val acc` | Overfitting | Naikkan `MAX_PER_CLASS`, tambah augmentasi, naikkan Dropout |
| Akurasi test sangat tinggi (>98%) tapi foto HP salah | Model mengenali gaya foto (latar putih = kertas), bukan objeknya | Tambah foto asli dari HP ke dataset; kurangi dominasi satu sumber per kelas |
| Satu kelas recall rendah di confusion matrix | Kelas itu mirip kelas lain (kardus vs kertas) | Lihat gambar yang salah, bersihkan label, atau ubah `INCLUDE_CARDBOARD_AS_PAPER` |
| `Kelas [...] tidak ditemukan` | Nama folder sumber tidak ada di `CLASS_MAP` | Lihat daftar "Diabaikan" di Langkah 3, tambahkan namanya |
| Out of memory | Batch terlalu besar | `BATCH_SIZE = 16` |

**Keterbatasan yang jujur disebut saat interview:** dataset berasal dari dua sumber dengan gaya foto berbeda; kelas "organic" hanya dari satu sumber sehingga ada risiko model belajar gaya foto; foto di dunia nyata (gelap, ramai, banyak objek) bisa turun akurasinya dibanding angka test.